# Smoke Test

Runs the full pipeline end-to-end with **3 steps / 5 examples** to verify
everything works before committing to the 4.5-hour full run.

**Setup:** GPU T4 x2 · Internet On · expected runtime ~5 min

In [ ]:
# ── 1. Clone repo ─────────────────────────────────────────────────────────
import shutil, os, sys, subprocess

os.chdir('/kaggle/working')
shutil.rmtree('/kaggle/working/Fine-tuning-and-GRPO-on-Qwen', ignore_errors=True)
subprocess.run(['git', 'clone',
    'https://github.com/202520030411/Fine-tuning-and-GRPO-on-Qwen.git',
    '/kaggle/working/Fine-tuning-and-GRPO-on-Qwen'], check=True)

REPO = '/kaggle/working/Fine-tuning-and-GRPO-on-Qwen'
os.chdir(REPO)
sys.path.insert(0, REPO)
os.environ['HF_HOME']            = '/kaggle/working/.hf'
os.environ['HF_DATASETS_CACHE']  = '/kaggle/working/.hf/datasets'
os.environ['TRANSFORMERS_CACHE'] = '/kaggle/working/.hf/transformers'
print('Repo ready')

In [ ]:
# ── 2. Install dependencies ───────────────────────────────────────────────
import subprocess
subprocess.run(['pip', 'install', '-q',
    'transformers>=4.51.0', 'peft>=0.11.1', 'datasets>=2.19.0',
    'accelerate', 'tqdm', 'typer', 'sentencepiece', 'safetensors',
    'matplotlib', 'trl>=0.12.0', 'tiktoken'
], check=True)
print('Done')

In [ ]:
# ── 3. Cache Qwen3-0.6B-Base ─────────────────────────────────────────────
from transformers import AutoModelForCausalLM, AutoTokenizer
AutoTokenizer.from_pretrained('Qwen/Qwen3-0.6B-Base')
AutoModelForCausalLM.from_pretrained('Qwen/Qwen3-0.6B-Base')
print('Model cached')

In [ ]:
# ── 4. Prepare datasets ───────────────────────────────────────────────────
!python scripts/prepare_gsm8k.py
!python scripts/prepare_svamp.py --limit 500

In [ ]:
# ── 5. Run smoke test ─────────────────────────────────────────────────────
# Tests the full pipeline (DoRA-SFT, LoRA-SFT, GRPO, eval, MMLU,
# prompt comparison, reasoning, analyze) with 3 steps / 5 examples.
# Outputs are saved to a temp directory and printed below.
import os, sys
ret = os.system('python scripts/smoke_test.py --keep-tmp')
if ret != 0:
    print('\n*** SMOKE TEST FAILED — do NOT proceed to full run ***')
    sys.exit(1)
else:
    print('\n✓ All checks passed — safe to run run_kaggle.ipynb')